# REST API Walkthrough - FastAPI

Demonstrates every endpoint on the **FastAPI** service (`http://localhost:5050`)
using plain `requests` calls. No Python library imports from `airgap_geo` are used

- all interactions go through the HTTP API.

Interactive Swagger docs are available at `http://localhost:5050/docs`.

## Prerequisites

| Service           | Default port | Role                      |
| ----------------- | ------------ | ------------------------- |
| **FastAPI (API)** | 5050         | Unified REST API          |
| Nominatim         | 8080         | Forward geocoding backend |
| Photon            | 2322         | Reverse geocoding backend |
| OSRM + HAProxy    | 80           | Routing backend           |
| postcodes.io      | 8000         | UK postcode backend       |

See **Start Services** below for Docker commands.


______________________________________________________________________

## Start Services

Before running this notebook, make sure the required Docker services are up.
From the **project root**, run:

```bash
make up                  # starts all services (including FastAPI)
```

Check running containers with `make ps`. See the main
[README](../README.md#docker-services) for all available profiles and
configuration details.


## Setup - Imports


In [ ]:
import json
import pprint

import pandas as pd
import requests

API_BASE = "http://localhost:5050"
print(f"API base URL: {API_BASE}")

______________________________________________________________________

## 1 - Health Check (`GET /health`)

The `/health` endpoint probes all four upstream backends concurrently and returns
per-service status, latency, and an overall status of `healthy`, `degraded`, or
`unhealthy`.


In [ ]:
resp = requests.get(f"{API_BASE}/health", timeout=10)
health = resp.json()

print(f"Overall status: {health['status']}\n")

rows = []
for name, info in health["services"].items():
    rows.append(
        {
            "Service": name,
            "Status": info["status"],
            "Latency (ms)": info.get("latency_ms", "-"),
            "Detail": info.get("detail") or "-",
        }
    )

pd.DataFrame(rows).set_index("Service")

______________________________________________________________________

## 2 - Geocoding (`GET /geocode?q=...`)

A single endpoint handles both forward geocoding (place name / postcode) and
reverse geocoding (lat,lon string). The response is a normalised `GeocodeResult`.


In [ ]:
# Forward geocode: place name
resp_fwd = requests.get(
    f"{API_BASE}/geocode", params={"q": "10 Downing Street, London"}, timeout=10
)
print(f"GET /geocode?q=10 Downing Street, London  [{resp_fwd.status_code}]\n")
pprint.pprint(resp_fwd.json())

In [ ]:
# Reverse geocode: coordinate string
resp_rev = requests.get(
    f"{API_BASE}/geocode", params={"q": "51.5007,-0.1246"}, timeout=10
)
print(f"GET /geocode?q=51.5007,-0.1246  [{resp_rev.status_code}]\n")
pprint.pprint(resp_rev.json())

______________________________________________________________________

## 3 - Routing (`POST /route`)

The route endpoint accepts a JSON body with origin, destination, profile, and
optional enhanced parameters (waypoints, steps, alternatives, annotations,
exclude, geometry format, etc.).


In [ ]:
# Basic A-to-B driving route
basic_body = {
    "origin": {"lat": 51.5308, "lon": -0.1238},
    "destination": {"lat": 51.4952, "lon": -0.1441},
    "profile": "driving",
}

resp_route = requests.post(f"{API_BASE}/route", json=basic_body, timeout=10)
print(f"POST /route (basic)  [{resp_route.status_code}]\n")

data = resp_route.json()
print(f"Profile     : {data['profile']}")
print(f"Origin      : {data['origin']}")
print(f"Destination : {data['destination']}")
if data["routes"]:
    r = data["routes"][0]
    print(f"Distance    : {r['distance_m'] / 1000:.2f} km")
    print(f"Duration    : {r['duration_s'] / 60:.1f} min")
    print(f"Legs        : {len(r['legs'])}")

In [ ]:
# Enhanced route: waypoints, steps, alternatives, and annotations
enhanced_body = {
    "origin": {"lat": 51.5308, "lon": -0.1238},
    "destination": {"lat": 51.4952, "lon": -0.1441},
    "waypoints": [{"lat": 51.5134, "lon": -0.0889}],
    "profile": "driving",
    "steps": True,
    "alternatives": False,
    "annotations": ["duration", "distance"],
}

resp_enh = requests.post(f"{API_BASE}/route", json=enhanced_body, timeout=10)
print(f"POST /route (enhanced)  [{resp_enh.status_code}]\n")

data_enh = resp_enh.json()
route_0 = data_enh["routes"][0]
print(f"Waypoints (input)   : {data_enh['waypoints']}")
print(f"Snapped waypoints   : {data_enh['snapped_waypoints']}")
print(f"Legs                : {len(route_0['legs'])}")
print(f"Steps in first leg  : {len(route_0['legs'][0]['steps'])}")

ann = route_0["legs"][0].get("annotation")
if ann:
    print(f"Annotation segments : {len(ann.get('duration', []))}")

In [ ]:
# Turn-by-turn steps from first leg
step_rows = []
for s in route_0["legs"][0]["steps"][:15]:
    m = s.get("manoeuvre") or {}
    step_rows.append(
        {
            "Road": s.get("name") or "(unnamed)",
            "Manoeuvre": m.get("type", "-"),
            "Modifier": m.get("modifier") or "-",
            "Distance (m)": round(s["distance_m"], 0),
            "Duration (s)": round(s["duration_s"], 0),
        }
    )

pd.DataFrame(step_rows)

In [ ]:
# Alternative routes
alt_body = {
    "origin": {"lat": 51.5308, "lon": -0.1238},
    "destination": {"lat": 51.4952, "lon": -0.1441},
    "profile": "driving",
    "alternatives": 3,
}

resp_alt = requests.post(f"{API_BASE}/route", json=alt_body, timeout=10)
data_alt = resp_alt.json()

print(f"Routes returned: {len(data_alt['routes'])}\n")

alt_rows = []
for i, r in enumerate(data_alt["routes"]):
    alt_rows.append(
        {
            "Route": f"Route {i + 1}",
            "Distance (km)": round(r["distance_m"] / 1000, 2),
            "Duration (min)": round(r["duration_s"] / 60, 1),
        }
    )

pd.DataFrame(alt_rows).set_index("Route")

______________________________________________________________________

## 4 - Postcode Lookup (`GET /postcodes/{postcode}`, `GET /outcodes/{outcode}`)

Full UK postcode and outcode (district) lookups.


In [ ]:
resp_pc = requests.get(f"{API_BASE}/postcodes/SW1A 2AA", timeout=10)
print(f"GET /postcodes/SW1A 2AA  [{resp_pc.status_code}]\n")
pc_data = resp_pc.json()
pprint.pprint(pc_data)

In [ ]:
resp_oc = requests.get(f"{API_BASE}/outcodes/SW1A", timeout=10)
print(f"GET /outcodes/SW1A  [{resp_oc.status_code}]\n")
oc_data = resp_oc.json()
pprint.pprint(oc_data)

In [ ]:
# Summary table
_FIELDS = [
    "postcode",
    "outcode",
    "latitude",
    "longitude",
    "admin_district",
    "admin_ward",
    "region",
    "country",
]

rows = []
for field in _FIELDS:
    rows.append(
        {
            "Field": field,
            "SW1A 2AA (postcode)": pc_data.get(field, "-"),
            "SW1A (outcode)": oc_data.get(field, "-"),
        }
    )

pd.DataFrame(rows).set_index("Field")

______________________________________________________________________

## 5 - Error Responses

The API returns standard HTTP error codes with a JSON `detail` field:

- **404** - resource not found (e.g. invalid postcode)
- **422** - validation error (e.g. unsupported routing profile)


In [ ]:
# 404: invalid postcode
resp_404 = requests.get(f"{API_BASE}/postcodes/ZZ9 9ZZ", timeout=10)
print(f"GET /postcodes/ZZ9 9ZZ  [{resp_404.status_code}]")
print(json.dumps(resp_404.json(), indent=2))

In [ ]:
# 422: unsupported routing profile
resp_422 = requests.post(
    f"{API_BASE}/route",
    json={
        "origin": {"lat": 51.5, "lon": -0.1},
        "destination": {"lat": 52.5, "lon": -1.9},
        "profile": "flying",
    },
    timeout=10,
)
print(f"POST /route (profile=flying)  [{resp_422.status_code}]")
print(json.dumps(resp_422.json(), indent=2))

______________________________________________________________________

## Stop Services

To stop containers, use `make down` from the project root. See the
[README](../README.md#docker-services) for details.
